# Setup

## Load packages

In [1]:
# Load up necessary packages. 
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Check GPU availability. 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Import my custom utils.
import utils

Using device: cuda
GPU: NVIDIA GeForce RTX 3090


## Load results

In [2]:
INPUT_DIR = "/tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/ml_inputs"

COMPARISON_TABLE = (
    "/tscc/nfs/home/kflanagan/projects/plip_plop/"
    "machine_learning_prototype/encode_initial_30_comparisons.tsv"
)

comparison_table = pd.read_csv(COMPARISON_TABLE, sep="\t")

# Running pytorch

## Data setup

### Create metadata

In [3]:
all_signals = []
all_metadata = []

for comparison_id, row in comparison_table.iterrows():
    experiment_A = row["experiment_A"]
    experiment_B = row["experiment_B"]

    npz_file = os.path.join(
        INPUT_DIR,
        f"{experiment_A}_{experiment_B}.npz"
    )

    if not os.path.exists(npz_file):
        print(f"Missing: {npz_file}")
        continue

    data = np.load(npz_file)

    # Load the already log-transformed, input-corrected signal.
    raw_signals = data["signals"].astype(np.float32)

    # Calculate signal strength before normalization.
    channel_totals = np.abs(raw_signals).sum(axis=2)
    total_signal = channel_totals.sum(axis=1)

    # Normalize each RBP independently by its L2 norm.
    channel_norms = np.sqrt((raw_signals ** 2).sum(axis=2, keepdims=True))

    signals = np.divide(
        raw_signals,
        channel_norms,
        out=np.zeros_like(raw_signals),
        where=channel_norms > 0
    )

    n_windows = len(signals)

    all_signals.append(signals)

    metadata = pd.DataFrame({
        "comparison_id": comparison_id,
        "category": row["category"],
        "experiment_A": experiment_A,
        "experiment_B": experiment_B,
        "chrom": data["chrom"],
        "start": data["start"],
        "end": data["end"],
        "strand": data["strand"],
        "region_id": data["region_id"],
        "block_number": data["block_number"],
        "signal_A": channel_totals[:, 0],
        "signal_B": channel_totals[:, 1],
        "total_signal": total_signal
    })

    all_metadata.append(metadata)

Missing: /tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/ml_inputs/FUS_HepG2_ENCSR464OSH_TAF15_HepG2_ENCSR841EQA.npz
Missing: /tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/ml_inputs/PUM1_K562_ENCSR308YNT_PUM2_K562_ENCSR661ICQ.npz
Missing: /tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/ml_inputs/PCBP1_HepG2_ENCSR256CHX_PCBP2_HepG2_ENCSR339FUY.npz
Missing: /tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/ml_inputs/TIA1_HepG2_ENCSR623VEQ_TIAL1_HepG2_ENCSR322HHA.npz
Missing: /tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/ml_inputs/SAFB_K562_ENCSR484LAB_SAFB2_K562_ENCSR943MHU.npz
Missing: /tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/ml_inputs/CSTF2_HepG2_ENCSR384MWO_CSTF2T_HepG2_ENCSR919HSE.npz
Missing: /tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/ml_inputs/IGF2BP1_K562_ENCSR975KIR_IGF2BP2_K562_ENCSR062NNB.npz
Missing: /tscc/lustre/ddn/scratch/kflanagan/plip_plop_results/ml_inputs/FMR1_K562_ENCSR331VNX_FXR1_K562_ENCSR774RFN.npz
Missing: /tscc/lust

In [4]:
signals = np.concatenate(all_signals, axis=0)
metadata = pd.concat(all_metadata, ignore_index=True)

ValueError: need at least one array to concatenate

### Weighting

In [ ]:
# Set weighting alpha to 0.5 (square root).

alpha = 0.5

# Create the comparison weights.

comparison_counts["comparison_weight"] = (
    comparison_counts["n_windows"] ** (alpha - 1)
)

# Create a dictionary for quickly grabbing each weight for each comparison.

weight_map = dict(zip(
    comparison_counts["comparison_id"],
    comparison_counts["comparison_weight"]
))

# Add comparison weights to metadata.

metadata["comparison_weight"] = metadata["comparison_id"].map(weight_map)

# Use signal magnitude for signal weighting.
# The signal has already been log-transformed during IN correction, so do not log-transform again.

metadata["signal_weight"] = np.abs(metadata["total_signal"])

# Normalize signal weights within each comparison.

metadata["signal_weight"] = (
    metadata["signal_weight"] /
    metadata.groupby("comparison_id")["signal_weight"].transform("mean")
)

# Create the final weight.

metadata["weight"] = (
    metadata["comparison_weight"] *
    metadata["signal_weight"]
)

## Train/val split setup. 

In [ ]:
# Create a combined identifier from comparison and region. 
metadata["region_key"] = (
    metadata["comparison_id"].astype(str)
    + "_"
    + metadata["region_id"].astype(str)
)

In [ ]:
# Setup random seed. 
rng = np.random.default_rng(42)

# Initialize masking vector. 
train_mask = np.zeros(len(metadata), dtype=bool)
val_mask = np.zeros(len(metadata), dtype=bool)

# Build mask. 
for comparison_id in metadata["comparison_id"].unique():
    comparison_rows = metadata["comparison_id"] == comparison_id

    comparison_regions = (
        metadata.loc[comparison_rows, "region_key"]
        .unique()
        .copy()
    )

    rng.shuffle(comparison_regions)

    split = int(len(comparison_regions) * 0.8)

    train_regions = comparison_regions[:split]
    val_regions = comparison_regions[split:]

    train_mask |= metadata["region_key"].isin(train_regions)
    val_mask |= metadata["region_key"].isin(val_regions)

In [ ]:
# Apply mask to the signals.  
train_signals = signals[train_mask]
val_signals = signals[val_mask]

# apply mask to the metadata. 
train_metadata = metadata.loc[train_mask].reset_index(drop=True)
val_metadata = metadata.loc[val_mask].reset_index(drop=True)

print("Training windows:", len(train_signals))
print("Validation windows:", len(val_signals))

## Setup data loaders. 

In [ ]:
# Create special data class for machine learning. 
class RBPWindowDataset(Dataset):
    def __init__(self, signals, weights):
        self.signals = torch.tensor(signals, dtype=torch.float32)
        self.weights = torch.tensor(weights, dtype=torch.float32)

    def __len__(self):
        return(len(self.signals))

    def __getitem__(self, idx):
        return(self.signals[idx], self.weights[idx])

train_weights = train_metadata["weight"].to_numpy(dtype=np.float32)
val_weights = val_metadata["weight"].to_numpy(dtype=np.float32)

train_dataset = RBPWindowDataset(train_signals, train_weights)
val_dataset = RBPWindowDataset(val_signals, val_weights)

train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)

In [ ]:
batch_signals, batch_weights = next(iter(train_loader))

print("Signals:", batch_signals.shape)
print("Weights:", batch_weights.shape)
print("First few weights:", batch_weights[:10])

## Model setup

In [ ]:
# Set model as autoencode
model = utils.SimpleAutoencoder(latent_dim=64).to(device)

# Select mean square error. 
criterion = nn.MSELoss()

# Adam optimizer (what is adam?)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

## Run model

In [ ]:
train_losses = []
val_losses = []

best_val_loss = float("inf")
best_model_state = None

num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    total_train_loss = 0

    for batch_signals, batch_weights in train_loader:
        batch_signals = batch_signals.to(device)
        batch_weights = batch_weights.to(device)
    
        optimizer.zero_grad()
    
        reconstruction = model(batch_signals)
        loss = utils.weighted_mse_loss(
            reconstruction,
            batch_signals,
            batch_weights
        )
    
        loss.backward()
        optimizer.step()
    
        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_loader)

    model.eval()
    total_val_loss = 0
    
    with torch.no_grad():
        for batch_signals, batch_weights in val_loader:
            batch_signals = batch_signals.to(device)
            batch_weights = batch_weights.to(device)
    
            reconstruction = model(batch_signals)
    
            loss = utils.weighted_mse_loss(
                reconstruction,
                batch_signals,
                batch_weights
            )
    
            total_val_loss += loss.item()

    average_val_loss = total_val_loss / len(val_loader)

    train_losses.append(average_train_loss)
    val_losses.append(average_val_loss)

    if average_val_loss < best_val_loss:
        best_val_loss = average_val_loss
        best_model_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }

    print(
        f"Epoch {epoch + 1}: "
        f"train = {average_train_loss:.6f}, "
        f"validation = {average_val_loss:.6f}"
    )

In [ ]:
# Loads best model, not just last model. 
model.load_state_dict(best_model_state)
model = model.to(device)

# Save model for later use. 
torch.save(model.state_dict(), "/tscc/nfs/home/kflanagan/scratch/plip_plop_results/many_IN_RBP_baseline.pt")